# Colab Setup

**For Colab users:** Uncomment and run the cell below to mount Drive.  
**For local users:** Skip this cell.

In [ ]:
# Uncomment for Colab:
#from google.colab import drive
#drive.mount('/content/drive')

## Setup: Dependency Verification and Installation

This cell checks for the presence of essential Python libraries (like `numpy`, `torch`, `musdb`, `nbformat`, etc.).
If any required library is not found, it attempts to install it automatically using `pip`.
It also verifies the availability of PyTorch with CUDA, which is crucial for GPU-accelerated training.

In [ ]:
# --- 1. Verify and install dependencies ---
print("Verifying and installing missing packages if necessary...")

packages_to_check = [
    'numpy', 'matplotlib', 'librosa', 'tqdm', 'sklearn', 'stempeg', 'torch', 'torchvision', 'torchaudio', 'musdb'
]

for package in packages_to_check:
    try:
        __import__(package)
        print(f"  ✅ {package} is installed.")
    except ImportError:
        print(f"  ❌ {package} is NOT installed. Attempting to install...")
        try:
            import sys
            import subprocess
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])
            __import__(package)
            print(f"  ✅ {package} is now installed.")
        except Exception as e:
            print(f"  ❌ Failed to install {package}: {e}")

# Special check for PyTorch CUDA
print("\n--- PyTorch CUDA status ---")
try:
    import torch
    if torch.cuda.is_available():
        print(f"  ✅ PyTorch with CUDA (version {torch.version.cuda}) is available.")
        print(f"     CUDA Device Name: {torch.cuda.get_device_name(0)}")
    else:
        print("  ⚠️ PyTorch is installed, but CUDA is NOT available.")
except ImportError:
    print("  ❌ PyTorch is not installed.")

print("Verification complete.")

## Imports and Environment Setup

- Import required libraries (torch, numpy, matplotlib, etc.)

- Set device (CPU/GPU)

In [ ]:
import sys
from pathlib import Path
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from IPython.display import Audio, display
import importlib

# Detect environment and set Project Root
try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/Final_Project_Deep_Learning')
    print(f"✅ Colab Project Root: {PROJECT_ROOT}")
else:
    # Local: use current working directory
    PROJECT_ROOT = Path.cwd()
    if not (PROJECT_ROOT / 'modelA.ipynb').exists():
        for p in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
            if (p / 'modelA.ipynb').exists():
                PROJECT_ROOT = p
                break

os.chdir(PROJECT_ROOT)

# Define data and checkpoint directories
DATA_DIR = PROJECT_ROOT / "data"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
DATA_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Add to sys.path for imports
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Import project modules
import models.utils as utils
importlib.reload(utils)
from models import model_A as ma
importlib.reload(ma)

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

# Device setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"\nConfiguration Complete:")
print(f"   - Device: {device}")
print(f"   - Working Directory: {os.getcwd()}")
print(f"   - Data Directory: {DATA_DIR}")
print(f"   - Checkpoints Directory: {CHECKPOINT_DIR}")

## MUSDB18 Setup

**Quick Start:**
1. Download MUSDB18
2. Extract to project folder as `musdb18/`
3. Run preprocessing below

**Expected structure:**
```
musdb18/
  train/     (~100 songs)
  valid/     (~14 songs)
  test/      (~50 songs)
```

In [ ]:
# ============================================================================
# MUSDB18 PATH CONFIGURATION
# ============================================================================
# Auto-detect musdb18 folder in project directory
MUSDB18_PATH = PROJECT_ROOT / "musdb18"

if not MUSDB18_PATH.exists():
    print("⚠️  MUSDB18 folder not found!")
    print(f"   Expected location: {MUSDB18_PATH}")
    print("")
    print("📥 Download MUSDB18:")
    print("   1. Register at https://zenodo.org/record/1438122")
    print("   2. Download MUSDB18-HQ.zip (~22GB)")
    print("   3. Extract to project folder as 'musdb18'")
    print("")
    MUSDB18_PATH = None
else:
    # Check for required subfolders
    train_dir = MUSDB18_PATH / "train"
    valid_dir = MUSDB18_PATH / "valid"
    test_dir = MUSDB18_PATH / "test"

    if train_dir.exists() and test_dir.exists():
        num_train = len(list(train_dir.iterdir()))
        num_valid = len(list(valid_dir.iterdir())) if valid_dir.exists() else 0
        num_test = len(list(test_dir.iterdir()))
        print(f"✅ MUSDB18 dataset found: {MUSDB18_PATH}")
        print(f"   Train: {num_train} tracks")
        print(f"   Valid: {num_valid} tracks")
        print(f"   Test: {num_test} tracks")
    else:
        print(f"⚠️  Found musdb18 folder but missing train/test subfolders")
        print(f"   Path: {MUSDB18_PATH}")
        MUSDB18_PATH = None

# Initialize file lists (will be populated after preprocessing)
mix_files_stage1 = mix_files_stage2 = tgt_files_stage1 = tgt_files_stage2 = []

## Data Preprocessing

Preprocess MUSDB18 into 8-second waveform chunks for training, validation, and testing.

In [ ]:
# ============================================================================
# DATA PREPROCESSING & PATH SETUP
# ============================================================================
import os
import shutil
import time
import zipfile
from pathlib import Path
from tqdm import tqdm

# ============================================================================
# 1. SETUP PATHS
# ============================================================================
# Define where your zip is on Drive (Update this path if needed!)
drive_zip_path = PROJECT_ROOT / "data.zip"

# Define where we want the data to live (Fast Local Disk)
local_extract_root = Path("/content/local_data")
local_data_dir = local_extract_root / "data"

# ============================================================================
# 2. DISK SPACE CHECK & EXTRACTION
# ============================================================================
if IN_COLAB:
    print(f"🔍 Checking for local data at: {local_data_dir}")

    # Check if data already exists to avoid re-unzipping
    if not local_data_dir.exists():
        print("🚀 Data not found locally. Starting extraction...")
        print(f"📂 Source: {drive_zip_path}")
        print(f"📂 Destination: {local_extract_root}")

        # Create destination folder
        local_extract_root.mkdir(parents=True, exist_ok=True)

        if drive_zip_path.exists():
            t0 = time.time()
            print("⏳ Unzipping from Drive with progress tracking...")

            try:
                with zipfile.ZipFile(drive_zip_path, 'r') as zip_ref:
                    # Get list of files
                    file_list = zip_ref.namelist()

                    # Extract with progress bar
                    for file in tqdm(file_list, desc="📦 Extracting", unit="file"):
                        zip_ref.extract(file, local_extract_root)

                t_final = time.time() - t0
                print(f"✅ Unzip complete in {t_final/60:.1f} minutes!")
                DATA_DIR = local_data_dir
            except Exception as e:
                print(f"❌ Unzip failed: {e}")
                print("   Falling back to Drive path (slow)")
                DATA_DIR = PROJECT_ROOT / "data"
        else:
            print(f"❌ Error: Could not find {drive_zip_path}")
            print("   Please ensure 'data.zip' is uploaded to your Drive project folder.")
            DATA_DIR = PROJECT_ROOT / "data" # Fallback (slow)

    else:
        print("✅ Fast local data already exists! Skipping unzip.")
        DATA_DIR = local_data_dir

    print(f"📍 DATA_DIR set to: {DATA_DIR}")

    # Check remaining disk space
    total, used, free = shutil.disk_usage("/")
    print(f"💾 Disk Space: {free // (2**30)} GB free / {total // (2**30)} GB total")

else:
    # Local PC
    DATA_DIR = PROJECT_ROOT / "data"
    print(f"💻 Running locally. Using repo data: {DATA_DIR}")

# ============================================================================
# 3. CONFIGURE UTILS
# ============================================================================
SAMPLE_RATE = 22050
CHUNK_DURATION = 8.0
CHUNK_OVERLAP = 4.0

if MUSDB18_PATH:
    utils.MUSDB_SPLITS = {
        'train': MUSDB18_PATH / 'train',
        'val': MUSDB18_PATH / 'valid',
        'test': MUSDB18_PATH / 'test',
    }
    utils.DATA_DIR = DATA_DIR
    utils.SAMPLE_RATE = SAMPLE_RATE
    utils.CHUNK_DURATION = CHUNK_DURATION
    utils.CHUNK_OVERLAP = CHUNK_OVERLAP

    # Since files are unzipped, these functions will see them and skip generation
    utils.process_stage1()
    utils.process_stage2()

    print(f"\n{'='*70}")
    print("🎯 SYSTEM READY")
    print(f"{'='*70}")
else:
    print("⚠️  MUSDB18_PATH not set - skipping preprocessing")

## Model A: Two Architectures for Comparison

**Model A (LSTM)** 1️⃣: Sequential bidirectional LSTM with masking output.

**Model A (U-Net)** 2️⃣: 2D CNN encoder-decoder with skip connections.

In [ ]:
print("="*70)
print("MODEL A ARCHITECTURES")
print("="*70)

# Quick architecture preview
lstm_preview, _, _, _ = utils.initialize_model_a_lstm(device)
unet_preview, _, _, _ = utils.initialize_model_a_unet(device)

lstm_params = sum(p.numel() for p in lstm_preview.parameters())
unet_params = sum(p.numel() for p in unet_preview.parameters())

print("\n1️⃣ Model A (LSTM):")
print(f"   Parameters: {lstm_params:,}")
print(f"   Type: Bidirectional LSTM with masking")

print("\n2️⃣ Model A (U-Net):")
print(f"   Parameters: {unet_params:,}")
print(f"   Type: 2D CNN encoder-decoder")

# Clean up preview models
del lstm_preview, unet_preview
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n" + "="*70)

## Train Both Models - Stage 1

Sequential training: LSTM first, then U-Net

In [ ]:
# ============================================================================
# TRAIN BOTH MODEL A ARCHITECTURES - STAGE 1 (2 -> 1)
# ============================================================================
import torch.optim as optim
import torch.nn as nn
from models import model_A as ma

SKIP_TRAINING_STAGE1 = False
CHUNK_DURATION = 8.0

print(f"\n{'='*70}")
print('STAGE 1 TRAINING: 2 → 1')
print(f"{'='*70}\n")

# Checkpoints
ckpt_lstm_s1 = CHECKPOINT_DIR / f'model_a_lstm_stage1_{CHUNK_DURATION:.0f}s.pth'
ckpt_unet_s1 = CHECKPOINT_DIR / f'model_a_unet_stage1_{CHUNK_DURATION:.0f}s.pth'

# Auto-skip
skip_lstm_s1 = SKIP_TRAINING_STAGE1 or ckpt_lstm_s1.exists()
skip_unet_s1 = SKIP_TRAINING_STAGE1 or ckpt_unet_s1.exists()

# --- 1. Train LSTM (FAST CONFIG) ---
print('1️⃣ Model A (LSTM) - Stage 1')
print('-' * 70)
if ckpt_lstm_s1.exists():
    print(f"⏭️  LSTM Stage 1 checkpoint found: {ckpt_lstm_s1.name}")

fast_lstm_config = utils.get_training_config_lstm()
fast_lstm_config['batch_size'] = 128 # LSTM is fine with big batches

model_lstm, processor_lstm, optimizer_lstm, loss_fn_lstm = utils.initialize_model_a_lstm(device)
hist_lstm_s1 = utils.train_model_stage(
    model=model_lstm,
    processor=processor_lstm,
    optimizer=optimizer_lstm,
    loss_fn=loss_fn_lstm,
    training_data_dir=DATA_DIR,
    stage='stage1',
    ckpt_path=ckpt_lstm_s1,
    device=device,
    train_config=fast_lstm_config,
    skip_training=skip_lstm_s1
)

# Free Memory
import gc
model_lstm = model_lstm.to('cpu')
del optimizer_lstm
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

# --- 2. Train U-Net (CORRECTED A100 CONFIG) ---
print('\n2️⃣ Model A (U-Net) - Stage 1')
print('-' * 70)
if ckpt_unet_s1.exists():
    print(f"⏭️  U-Net Stage 1 checkpoint found: {ckpt_unet_s1.name}")

# CONFIG: Use Batch Size 32 (Safe limit for massive spectrograms)
fast_unet_config = utils.get_training_config_unet()
fast_unet_config['batch_size'] = 32  # <--- CHANGED FROM 128 to 32

# MANUAL INITIALIZATION (Light Model)
print("📉 Initializing Optimized U-Net (32 filters, 5 layers)...")
model_unet = ma.TimeFrequencyDomainUNet(
    in_channels=1,
    out_channels=1,
    base_filters=32,
    num_layers=5,
    batchnorm=True,
    dropout=0.1
).to(device)

# Manual Optimizer Setup
processor_unet = utils.AudioProcessor(device=device)
optimizer_unet = optim.Adam(model_unet.parameters(), lr=fast_unet_config['learning_rate'])
loss_fn_unet = nn.MSELoss()

# Train
hist_unet_s1 = utils.train_model_stage(
    model=model_unet,
    processor=processor_unet,
    optimizer=optimizer_unet,
    loss_fn=loss_fn_unet,
    training_data_dir=DATA_DIR,
    stage='stage1',
    ckpt_path=ckpt_unet_s1,
    device=device,
    train_config=fast_unet_config,
    skip_training=skip_unet_s1
)

print(f"\n{'='*70}")
print('✅ STAGE 1 COMPLETE!')
print(f"{'='*70}")

## Stage 1 Test Evaluation

Compute loss on held-out test set (forward pass only) to assess generalization.


In [ ]:
# ============================================================================
# STAGE 1 TEST EVALUATION - FORWARD PASS ONLY
# ============================================================================

def evaluate_test_set(model, processor, test_data_dir, stage, loss_fn, device, sr=22050):
    """
    Compute loss on test set (forward pass only, no gradients).
    Assess generalization to unseen data.
    """
    test_loader = utils.get_data_loaders(test_data_dir, stage=stage, split='test', batch_size=32)

    model.eval()
    test_losses = []

    with torch.no_grad():
        for batch_idx, batch in enumerate(test_loader):
            mix = batch['mix']
            target = batch['tgt']

            # Check if data is already spectrograms (tuple) or waveforms (tensor)
            if isinstance(mix, tuple):
                # Already spectrograms - extract magnitude and move to device
                mix_mag = mix[0].to(device)
                tgt_mag = target[0].to(device)
                
                # Add channel dimension if needed (model expects 4D: batch, channel, freq, time)
                if mix_mag.dim() == 3:
                    mix_mag = mix_mag.unsqueeze(1)
                if tgt_mag.dim() == 3:
                    tgt_mag = tgt_mag.unsqueeze(1)
                
                mix_processed = mix_mag
                target_processed = tgt_mag
            else:
                # Waveforms - convert to spectrograms
                mix = mix.to(device)
                target = target.to(device)
                
                mix_spec = processor.to_spectrogram(mix)
                target_spec = processor.to_spectrogram(target)
                
                # Extract magnitude and add channel dimension
                mix_processed = mix_spec[0].unsqueeze(1) if mix_spec[0].dim() == 3 else mix_spec[0]
                target_processed = target_spec[0].unsqueeze(1) if target_spec[0].dim() == 3 else target_spec[0]

            # Forward pass (model outputs a mask)
            mask = model(mix_processed)
            
            # Ensure mask shape matches input
            if mask.shape != mix_processed.shape:
                mask = mask[:, :, :mix_processed.shape[2], :mix_processed.shape[3]]
            
            # Apply mask in LINEAR domain (same as training)
            est_linear = mask * torch.expm1(mix_processed)
            est_log = torch.log1p(est_linear)
            
            # Compute loss in LOG domain
            loss = loss_fn(est_log, target_processed)
            test_losses.append(loss.item())

            if (batch_idx + 1) % max(1, len(test_loader) // 5) == 0 or batch_idx == 0:
                print(f"  Batch {batch_idx+1}/{len(test_loader)} | Loss: {loss.item():.6f}")

    avg_test_loss = np.mean(test_losses)
    std_test_loss = np.std(test_losses)

    print(f"\n{'='*70}")
    print(f"TEST RESULTS:")
    print(f"  Mean Loss: {avg_test_loss:.6f}")
    print(f"  Std Loss:  {std_test_loss:.6f}")
    print(f"  Min Loss:  {min(test_losses):.6f}")
    print(f"  Max Loss:  {max(test_losses):.6f}")
    print(f"{'='*70}\n")

    return {'test_losses': test_losses, 'mean': avg_test_loss, 'std': std_test_loss}

# Evaluate both models on test set
print("\n" + "🧪 STAGE 1 TEST GENERALIZATION" + "\n")

# Ensure trained weights are loaded from checkpoints
model_lstm.to(device)
model_unet.to(device)

# Load LSTM weights if checkpoint exists
if ckpt_lstm_s1.exists():
    print(f"📥 Loading LSTM Stage 1 weights for evaluation from: {ckpt_lstm_s1.name}")
    checkpoint = torch.load(ckpt_lstm_s1, map_location=device, weights_only=False)
    model_lstm.load_state_dict(checkpoint['model_state_dict'])
    print(f"✅ LSTM weights loaded (trained for {checkpoint.get('epoch', '?')} epochs)")
else:
    print("⚠️ No LSTM checkpoint found - evaluating untrained model!")

# Load U-Net weights if checkpoint exists  
if ckpt_unet_s1.exists():
    print(f"📥 Loading U-Net Stage 1 weights for evaluation from: {ckpt_unet_s1.name}")
    checkpoint = torch.load(ckpt_unet_s1, map_location=device, weights_only=False)
    model_unet.load_state_dict(checkpoint['model_state_dict'])
    print(f"✅ U-Net weights loaded (trained for {checkpoint.get('epoch', '?')} epochs)")
else:
    print("⚠️ No U-Net checkpoint found - evaluating untrained model!")

print("\n" + "="*70)
print("🔍 EVALUATING LSTM MODEL")
print("="*70)
test_lstm_s1 = evaluate_test_set(model_lstm, processor_lstm, DATA_DIR,
                                  'stage1', loss_fn_lstm, device)

print("\n" + "="*70)
print("🔍 EVALUATING U-NET MODEL")
print("="*70)
test_unet_s1 = evaluate_test_set(model_unet, processor_unet, DATA_DIR,
                                  'stage1', loss_fn_unet, device)

# Plot train/val/test comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_lstm = range(1, len(hist_lstm_s1['train_loss']) + 1)
ax1.plot(epochs_lstm, hist_lstm_s1['train_loss'], 'o-', label='Train', linewidth=2)
ax1.plot(epochs_lstm, hist_lstm_s1['val_loss'], 's--', label='Val', linewidth=2)
ax1.axhline(test_lstm_s1['mean'], color='red', linestyle=':', linewidth=2, label=f"Test (μ={test_lstm_s1['mean']:.4f})")
ax1.fill_between(epochs_lstm,
                  test_lstm_s1['mean'] - test_lstm_s1['std'],
                  test_lstm_s1['mean'] + test_lstm_s1['std'],
                  alpha=0.2, color='red')
ax1.set_title('Model A (LSTM) - Stage 1', fontsize=12, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

epochs_unet = range(1, len(hist_unet_s1['train_loss']) + 1)
ax2.plot(epochs_unet, hist_unet_s1['train_loss'], 'o-', label='Train', linewidth=2)
ax2.plot(epochs_unet, hist_unet_s1['val_loss'], 's--', label='Val', linewidth=2)
ax2.axhline(test_unet_s1['mean'], color='red', linestyle=':', linewidth=2, label=f"Test (μ={test_unet_s1['mean']:.4f})")
ax2.fill_between(epochs_unet,
                  test_unet_s1['mean'] - test_unet_s1['std'],
                  test_unet_s1['mean'] + test_unet_s1['std'],
                  alpha=0.2, color='red')
ax2.set_title('Model A (U-Net) - Stage 1', fontsize=12, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Train/Val/Test Comparison - Stage 1', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Compare Training Results (Stage 1)

Side-by-side comparison of U-Net vs LSTM training curves for Stage 1.

In [ ]:
# ============================================================================
# PLOT TRAINING COMPARISON (STAGE 1)
# ============================================================================

# Load histories from checkpoints if not available
if not hist_lstm_s1:
    ckpt_lstm = CHECKPOINT_DIR / f"model_a_lstm_stage1_{CHUNK_DURATION:.0f}s.pth"
    hist_lstm_s1 = utils.load_training_history_from_checkpoint(ckpt_lstm)

if not hist_unet_s1:
    ckpt_unet = CHECKPOINT_DIR / f"model_a_unet_stage1_{CHUNK_DURATION:.0f}s.pth"
    hist_unet_s1 = utils.load_training_history_from_checkpoint(ckpt_unet)

# Plot comparison using utility function
utils.plot_model_comparison(hist_lstm_s1, hist_unet_s1,
                            title="Model A Comparison: LSTM vs U-Net - Stage 1")

## Stage 1 Evaluation (Spectrograms + Audio)

Compare LSTM vs U-Net separation quality on Stage 1 test samples.

In [ ]:
# ============================================================================
# STAGE 1 EVALUATION - LSTM vs U-NET - INTERACTIVE TEST SONG SELECTION
# ============================================================================

import librosa

# Move models back to device
model_lstm = model_lstm.to(device)
model_unet = model_unet.to(device)

# Extract a segment from a test song and run inference on both models
model_lstm.eval()
model_unet.eval()

SR = 22050
SEGMENT_START_SEC = 0  # Start at 0:00
SEGMENT_DURATION_SEC = 120  # 2 minutes (0:00 to 2:00)

# List all test tracks
if MUSDB18_PATH:
    test_root = MUSDB18_PATH / "test"
    test_tracks = sorted([p for p in test_root.iterdir() if p.is_dir()])
    
    print("\n" + "="*70)
    print("Available Test Tracks:")
    print("="*70)
    for i, track in enumerate(test_tracks):
        print(f"  [{i:2d}] {track.name}")
    
    # Get track selection from user or use default
    print(f"\n📝 Enter a track number [0-{len(test_tracks)-1}] or press Enter for default (2)")
    try:
        # Try to get user input, with timeout fallback
        import sys
        choice_str = ""
        try:
            # This will work in interactive notebooks
            choice_str = input("Select track: ").strip()
        except (EOFError, StopIteration):
            # Fallback if no input available
            choice_str = ""
        
        if choice_str:
            choice = int(choice_str)
        else:
            choice = 2  # Default to track 2 (The Sunshine Garcia Band)
        
        if not (0 <= choice < len(test_tracks)):
            print(f"⚠️  Invalid choice. Using default (2)")
            choice = 2
    except ValueError:
        print(f"⚠️  Invalid input. Using default (2)")
        choice = 2
    
    selected_track = test_tracks[choice]
    print(f"\n✅ Selected: Track {choice} - {selected_track.name}")
    
    # Load stems from the selected track
    print(f"📥 Loading stems...")
    try:
        stems_dict = utils.load_musdb_stems(selected_track)
        
        # STAGE 1: Use vocals + other only (matches training: vocals+other → other)
        mix_full = stems_dict['vocals'] + stems_dict['other']  # Only 2 stems for Stage 1
        other_full = stems_dict['other']  # Target: other/accompaniment stem
        
        print(f"   ℹ️  Stage 1 mixture: vocals + other (2 stems)")
        
        # Extract segment (0:00 to 2:00)
        start_sample = int(SEGMENT_START_SEC * SR)
        end_sample = int((SEGMENT_START_SEC + SEGMENT_DURATION_SEC) * SR)
        
        mix_segment = mix_full[start_sample:end_sample]
        other_segment = other_full[start_sample:end_sample]
        
        print(f"✅ Loaded: mix shape {mix_full.shape}, extracted segment {mix_segment.shape}")
        print(f"   Time range: {SEGMENT_START_SEC:.0f}s - {SEGMENT_START_SEC + SEGMENT_DURATION_SEC:.0f}s")
        
        # Helper function to run inference with chunking
        def predict_other(model, processor, mix_waveform, model_name):
            """Run inference with chunking (8s chunks, 4s overlap) and return predicted other stem"""
            CHUNK_LEN = 8.0  # seconds (matches training)
            OVERLAP_LEN = 4.0  # seconds (matches training)
            
            chunk_samples = int(CHUNK_LEN * SR)
            overlap_samples = int(OVERLAP_LEN * SR)
            stride_samples = chunk_samples - overlap_samples
            
            total_samples = len(mix_waveform)
            est_full = np.zeros_like(mix_waveform)
            weight_full = np.zeros_like(mix_waveform)
            
            print(f"   Chunking: {CHUNK_LEN}s chunks with {OVERLAP_LEN}s overlap (stride={stride_samples})")
            print(f"   Total samples: {total_samples}, chunk samples: {chunk_samples}")
            
            chunk_idx = 0
            pos = 0
            
            while pos < total_samples:
                # Extract chunk
                chunk_start = pos
                chunk_end = min(pos + chunk_samples, total_samples)
                mix_chunk = mix_waveform[chunk_start:chunk_end]
                
                # Pad if necessary (last chunk might be shorter)
                if len(mix_chunk) < chunk_samples:
                    mix_chunk = np.pad(mix_chunk, (0, chunk_samples - len(mix_chunk)), mode='constant')
                    pad_amount = chunk_samples - (chunk_end - chunk_start)
                    print(f"   Chunk {chunk_idx}: padded by {pad_amount} samples")
                else:
                    print(f"   Chunk {chunk_idx}: {chunk_start}-{chunk_end} samples")
                
                # Process chunk
                with torch.no_grad():
                    # Convert to spectrogram
                    mix_spec_tuple = processor.to_spectrogram(mix_chunk)
                    mix_mag = mix_spec_tuple[0]
                    mix_phase = mix_spec_tuple[1]
                    
                    # Ensure 2D
                    while mix_mag.dim() > 2:
                        mix_mag = mix_mag.squeeze(0)
                    
                    # Transpose to (freq, time) if needed
                    if mix_mag.shape[0] < mix_mag.shape[1]:
                        mix_mag = mix_mag.t()
                    
                    # Process through model
                    mix_input = mix_mag.unsqueeze(0).unsqueeze(0).to(device)  # (1, 1, freq, time)
                    mask = model(mix_input)
                    mask = mask.squeeze(0).squeeze(0)  # (freq, time)
                    
                    # Ensure mask matches input
                    if mask.shape != mix_mag.shape:
                        mask = mask[:mix_mag.shape[0], :mix_mag.shape[1]]
                    
                    # Apply mask
                    est_linear = mask * torch.expm1(mix_mag)
                    est_mag = torch.log1p(est_linear)
                    
                    # Reconstruct waveform
                    est_mag_np = est_mag.cpu().numpy()
                    mix_phase_np = mix_phase.numpy()
                    while mix_phase_np.ndim > 2:
                        mix_phase_np = mix_phase_np.squeeze(0)
                    
                    est_chunk = processor.to_waveform(est_mag_np, mix_phase_np)
                    
                    # Trim to actual chunk size (remove padding)
                    est_chunk = est_chunk[:chunk_end - chunk_start]
                
                # Add to output with Hann window weighting for overlap handling
                chunk_out_len = len(est_chunk)
                window = np.hanning(chunk_samples)[:chunk_out_len]
                
                est_full[chunk_start:chunk_start + chunk_out_len] += est_chunk * window
                weight_full[chunk_start:chunk_start + chunk_out_len] += window
                
                chunk_idx += 1
                pos += stride_samples
                
                if pos >= total_samples:
                    break
            
            # Normalize by weights (handle overlaps)
            est_full = np.divide(est_full, weight_full, where=weight_full > 0, out=est_full.copy())
            
            # Also return the full mix spectrogram for visualization
            with torch.no_grad():
                mix_spec_tuple = processor.to_spectrogram(mix_waveform)
                mix_mag = mix_spec_tuple[0]
                mix_phase = mix_spec_tuple[1]
                while mix_mag.dim() > 2:
                    mix_mag = mix_mag.squeeze(0)
                if mix_mag.shape[0] < mix_mag.shape[1]:
                    mix_mag = mix_mag.t()
                mix_mag_np = mix_mag.cpu().numpy()
                mix_phase_np = mix_phase.numpy()
                while mix_phase_np.ndim > 2:
                    mix_phase_np = mix_phase_np.squeeze(0)
            
            # Get est_mag from full reconstruction
            est_spec_tuple = processor.to_spectrogram(est_full)
            est_mag_full = est_spec_tuple[0]
            while est_mag_full.dim() > 2:
                est_mag_full = est_mag_full.squeeze(0)
            if est_mag_full.shape[0] < est_mag_full.shape[1]:
                est_mag_full = est_mag_full.t()
            est_mag_np = est_mag_full.cpu().numpy()
            
            return est_full, est_mag_np, mix_phase_np, mix_mag_np
        
        # Run inference with LSTM
        print("\n" + "="*70)
        print("🔍 EVALUATING LSTM MODEL")
        print("="*70)
        est_lstm, est_mag_lstm, phase, mix_mag_np = predict_other(
            model_lstm, processor_lstm, mix_segment, "LSTM"
        )
        print(f"✅ LSTM prediction shape: {est_lstm.shape}")
        
        # Run inference with U-Net
        print("\n" + "="*70)
        print("🔍 EVALUATING U-NET MODEL")
        print("="*70)
        est_unet, est_mag_unet, _, _ = predict_other(
            model_unet, processor_unet, mix_segment, "U-Net"
        )
        print(f"✅ U-Net prediction shape: {est_unet.shape}")
        
        # Display and play audio
        print("\n" + "="*70)
        print("SPECTROGRAMS COMPARISON")
        print("="*70)
        
        fig, axes = plt.subplots(3, 2, figsize=(16, 12))
        
        # Helper function to display spectrogram on axis
        def display_spec_on_axis(ax, spec_data, title, cmap='viridis'):
            """Display spec on matplotlib axis"""
            im = ax.imshow(spec_data, aspect='auto', origin='lower', cmap=cmap)
            ax.set_title(title, fontsize=12, fontweight='bold')
            ax.set_ylabel('Frequency (bins)')
            ax.set_xlabel('Time (frames)')
            plt.colorbar(im, ax=ax)
            return im
        
        # Mix spectrogram
        display_spec_on_axis(axes[0, 0], mix_mag_np, "Mix (Input)")
        axes[0, 1].axis('off')
        axes[0, 1].text(0.5, 0.5, f"Song: {selected_track.name}\nSegment: {SEGMENT_START_SEC:.0f}s - {SEGMENT_START_SEC + SEGMENT_DURATION_SEC:.0f}s",
                       ha='center', va='center', fontsize=12, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        # Target (other) spectrogram
        target_mag_tuple = processor_lstm.to_spectrogram(other_segment)
        target_mag = target_mag_tuple[0].numpy()
        # Handle extra dimensions if needed
        while target_mag.ndim > 2:
            target_mag = target_mag.squeeze(0)
        if target_mag.shape[0] < target_mag.shape[1]:
            target_mag = target_mag.T  # Ensure (freq, time)
        display_spec_on_axis(axes[1, 0], target_mag, "Target (Other/Accompaniment)")
        
        # LSTM prediction spectrogram
        display_spec_on_axis(axes[1, 1], est_mag_lstm, "LSTM Prediction")
        
        # U-Net prediction spectrogram
        display_spec_on_axis(axes[2, 0], est_mag_unet, "U-Net Prediction")
        axes[2, 1].axis('off')
        plt.suptitle(f"Stage 1 Inference: {selected_track.name}", fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
        
        # Play audio comparison
        print("\n" + "="*70)
        print("AUDIO PLAYBACK")
        print("="*70)
        
        print("\n🔊 Mix (Input):")
        display(Audio(mix_segment, rate=SR))
        
        print("\n🔊 Target (Other/Accompaniment):")
        display(Audio(other_segment, rate=SR))
        
        print("\n🔊 LSTM Prediction:")
        display(Audio(est_lstm, rate=SR))
        
        print("\n🔊 U-Net Prediction:")
        display(Audio(est_unet, rate=SR))
        
        print("\n" + "="*70)
        print("✅ INFERENCE COMPLETE")
        print("="*70)
        
    except Exception as e:
        print(f"❌ Error during inference: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⚠️  MUSDB18_PATH not set - unable to load test songs")

## Train Both Models - Stage 2

Curriculum step: Stage 2 uses 4→1 channels and continues from Stage 1 weights.

In [ ]:
# ============================================================================
# TRAIN BOTH MODEL A ARCHITECTURES - STAGE 2 (4 -> 1)
# ============================================================================
import torch.optim as optim
import torch.nn as nn
from models import model_A as ma  # <--- Essential import for manual init

STAGE2_ENABLED = True
SKIP_TRAINING_STAGE2 = False

# High-speed configurations
FAST_LSTM_CONFIG = utils.get_training_config_lstm()
FAST_LSTM_CONFIG['batch_size'] = 128

FAST_UNET_CONFIG = utils.get_training_config_unet()
FAST_UNET_CONFIG['batch_size'] = 32 # Safe batch size for A100 with massive spectrograms

# Checkpoints
ckpt_lstm_s2 = CHECKPOINT_DIR / f"model_a_lstm_stage2_{CHUNK_DURATION:.0f}s.pth"
ckpt_unet_s2 = CHECKPOINT_DIR / f"model_a_unet_stage2_{CHUNK_DURATION:.0f}s.pth"
ckpt_lstm_s1 = CHECKPOINT_DIR / f"model_a_lstm_stage1_{CHUNK_DURATION:.0f}s.pth"
ckpt_unet_s1 = CHECKPOINT_DIR / f"model_a_unet_stage1_{CHUNK_DURATION:.0f}s.pth"

hist_lstm_s2 = {}
hist_unet_s2 = {}

if not STAGE2_ENABLED:
    print("⏭️  Stage 2 training disabled (STAGE2_ENABLED = False)")
else:
    # --- LSTM ---
    print("1️⃣ Model A (LSTM) - Stage 2")
    print("-" * 70)

    # LSTM is standard, so utils.initialize is fine here
    model_lstm, processor_lstm, optimizer_lstm, loss_fn_lstm = utils.initialize_model_a_lstm(device)

    # Load Stage 1 weights
    if ckpt_lstm_s1.exists():
        print(f"📥 Loading Stage 1 weights from: {ckpt_lstm_s1.name}")
        checkpoint = torch.load(ckpt_lstm_s1, map_location=device)
        model_lstm.load_state_dict(checkpoint['model_state_dict'])
        print(f"✅ LSTM Stage 1 weights loaded successfully.")
    else:
        print("⚠️ LSTM Stage 1 checkpoint NOT found. Starting from scratch (not ideal).")

    hist_lstm_s2 = utils.train_model_stage(
        model=model_lstm,
        processor=processor_lstm,
        optimizer=optimizer_lstm,
        loss_fn=loss_fn_lstm,
        training_data_dir=DATA_DIR,
        stage="stage2",
        ckpt_path=ckpt_lstm_s2,
        device=device,
        train_config=FAST_LSTM_CONFIG, # Fast batch size
        skip_training=SKIP_TRAINING_STAGE2
    )

    # Free memory
    import gc
    model_lstm = model_lstm.to('cpu')
    del optimizer_lstm
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    # --- U-Net ---
    print("\n2️⃣ Model A (U-Net) - Stage 2")
    print("-" * 70)

    # 🛑 MANUAL INITIALIZATION (Must match Stage 1 structure exactly!)
    print("📉 Initializing Optimized U-Net for Stage 2 (32 filters, 5 layers)...")
    model_unet = ma.TimeFrequencyDomainUNet(
        in_channels=1,
        out_channels=1,
        base_filters=32,   # <--- MATCHES STAGE 1
        num_layers=5,      # <--- MATCHES STAGE 1
        batchnorm=True,
        dropout=0.1
    ).to(device)

    # Manual Optimizer Setup (since we skipped utils.init)
    processor_unet = utils.AudioProcessor(device=device)
    optimizer_unet = optim.Adam(model_unet.parameters(), lr=FAST_UNET_CONFIG['learning_rate'])
    loss_fn_unet = nn.MSELoss()

    # Load Stage 1 weights
    if ckpt_unet_s1.exists():
        print(f"📥 Loading Stage 1 weights from: {ckpt_unet_s1.name}")
        checkpoint = torch.load(ckpt_unet_s1, map_location=device)
        try:
            model_unet.load_state_dict(checkpoint['model_state_dict'])
            print(f"✅ U-Net Stage 1 weights loaded successfully.")
        except RuntimeError as e:
            print(f"❌ CRITICAL ERROR loading weights: {e}")
            print("   (This usually means the model structure in Stage 2 doesn't match Stage 1)")
            raise e
    else:
        print("⚠️ U-Net Stage 1 checkpoint NOT found. Starting from scratch.")

    hist_unet_s2 = utils.train_model_stage(
        model=model_unet,
        processor=processor_unet,
        optimizer=optimizer_unet,
        loss_fn=loss_fn_unet,
        training_data_dir=DATA_DIR,
        stage="stage2",
        ckpt_path=ckpt_unet_s2,
        device=device,
        train_config=FAST_UNET_CONFIG, # Fast batch size
        skip_training=SKIP_TRAINING_STAGE2
    )

print(f"\n{'='*70}")
print("✅ STAGE 2 COMPLETE!")
print(f"{'='*70}")

## Stage 2 Test Evaluation

Compute loss on held-out test set (forward pass only) to assess generalization.


In [ ]:
# ============================================================================
# STAGE 2 EVALUATION - LSTM vs U-NET - INTERACTIVE TEST SONG SELECTION
# ============================================================================
# Goal: Compare LSTM vs U-Net on the FULL mixture (Drums+Bass+Other+Vocals -> Vocals)

import librosa
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display
import torch

# --- 1. Load Stage 2 Models (Manual Init for U-Net) ---
print(f"\n{'='*70}")
print('RELOADING MODELS FOR STAGE 2 (Full Mix -> Vocals)')
print(f"{'='*70}")

# LSTM Stage 2
model_lstm_s2, processor_lstm_s2, _, _ = utils.initialize_model_a_lstm(device)
if ckpt_lstm_s2.exists():
    checkpoint = torch.load(ckpt_lstm_s2, map_location=device)
    model_lstm_s2.load_state_dict(checkpoint['model_state_dict'])
    print(f"✅ Loaded LSTM Stage 2 weights: {ckpt_lstm_s2.name}")
else:
    print(f"⚠️ LSTM Stage 2 checkpoint not found!")

# U-Net Stage 2 (Manual "Light" Init)
model_unet_s2 = ma.TimeFrequencyDomainUNet(
    in_channels=1,
    out_channels=1,
    base_filters=32,   # Match training config
    num_layers=5,      # Match training config
    batchnorm=True,
    dropout=0.1
).to(device)
processor_unet_s2 = utils.AudioProcessor(device=device) # Need separate processor instance

if ckpt_unet_s2.exists():
    checkpoint = torch.load(ckpt_unet_s2, map_location=device)
    model_unet_s2.load_state_dict(checkpoint['model_state_dict'])
    print(f"✅ Loaded U-Net Stage 2 weights: {ckpt_unet_s2.name}")
else:
    print(f"⚠️ U-Net Stage 2 checkpoint not found!")

model_lstm_s2.eval()
model_unet_s2.eval()

# --- 2. Interactive Selection & Inference ---
SR = 22050
SEGMENT_START_SEC = 0  # Start at 0:00
SEGMENT_DURATION_SEC = 120  # 2 minutes

if MUSDB18_PATH:
    test_root = MUSDB18_PATH / "test"
    test_tracks = sorted([p for p in test_root.iterdir() if p.is_dir()])
    
    print("\n" + "="*70)
    print("Available Test Tracks (Stage 2):")
    print("="*70)
    for i, track in enumerate(test_tracks):
        print(f"  [{i:2d}] {track.name}")
    
    print(f"\n📝 Enter a track number [0-{len(test_tracks)-1}] or press Enter for default (2)")
    try:
        import sys
        choice_str = ""
        try:
            choice_str = input("Select track: ").strip()
        except (EOFError, StopIteration):
            choice_str = ""
        
        if choice_str:
            choice = int(choice_str)
        else:
            choice = 2  # Default
        
        if not (0 <= choice < len(test_tracks)):
            print(f"⚠️  Invalid choice. Using default (2)")
            choice = 2
    except ValueError:
        print(f"⚠️  Invalid input. Using default (2)")
        choice = 2
    
    selected_track = test_tracks[choice]
    print(f"\n✅ Selected: Track {choice} - {selected_track.name}")
    
    # Load stems
    print(f"📥 Loading stems...")
    try:
        stems_dict = utils.load_musdb_stems(selected_track)
        
        # STAGE 2: Use FULL mixture (Vocals + Bass + Drums + Other)
        # Target: Vocals
        mix_full = stems_dict['mixture'] 
        target_full = stems_dict['vocals']
        
        print(f"   ℹ️  Stage 2 mixture: Full Band (4 stems)")
        print(f"   🎯 Target: Vocals")
        
        # Extract segment
        start_sample = int(SEGMENT_START_SEC * SR)
        end_sample = int((SEGMENT_START_SEC + SEGMENT_DURATION_SEC) * SR)
        
        mix_segment = mix_full[start_sample:end_sample]
        target_segment = target_full[start_sample:end_sample]
        
        print(f"✅ Loaded: mix shape {mix_full.shape}, extracted segment {mix_segment.shape}")
        
        # --- Helper Function (Copied for scope) ---
        def predict_stem(model, processor, mix_waveform):
            """Run inference with chunking (8s chunks, 4s overlap)"""
            CHUNK_LEN = 8.0 
            OVERLAP_LEN = 4.0 
            chunk_samples = int(CHUNK_LEN * SR)
            overlap_samples = int(OVERLAP_LEN * SR)
            stride_samples = chunk_samples - overlap_samples
            
            total_samples = len(mix_waveform)
            est_full = np.zeros_like(mix_waveform)
            weight_full = np.zeros_like(mix_waveform)
            
            chunk_idx = 0
            pos = 0
            
            # Progress bar simulation
            print(f"   Processing...", end="", flush=True)

            while pos < total_samples:
                if chunk_idx % 5 == 0: print(".", end="", flush=True)

                chunk_start = pos
                chunk_end = min(pos + chunk_samples, total_samples)
                mix_chunk = mix_waveform[chunk_start:chunk_end]
                
                # Pad
                if len(mix_chunk) < chunk_samples:
                    mix_chunk = np.pad(mix_chunk, (0, chunk_samples - len(mix_chunk)), mode='constant')
                
                with torch.no_grad():
                    # To Spectrogram
                    mix_spec_tuple = processor.to_spectrogram(mix_chunk)
                    mix_mag = mix_spec_tuple[0]
                    mix_phase = mix_spec_tuple[1]
                    
                    while mix_mag.dim() > 2: mix_mag = mix_mag.squeeze(0)
                    if mix_mag.shape[0] < mix_mag.shape[1]: mix_mag = mix_mag.t()
                    
                    mix_input = mix_mag.unsqueeze(0).unsqueeze(0).to(device)
                    
                    # Model Inference
                    mask = model(mix_input)
                    mask = mask.squeeze(0).squeeze(0)
                    
                    if mask.shape != mix_mag.shape:
                        mask = mask[:mix_mag.shape[0], :mix_mag.shape[1]]
                    
                    # Apply Mask
                    est_linear = mask * torch.expm1(mix_mag)
                    est_mag = torch.log1p(est_linear)
                    
                    # To Waveform
                    est_mag_np = est_mag.cpu().numpy()
                    mix_phase_np = mix_phase.numpy()
                    while mix_phase_np.ndim > 2: mix_phase_np = mix_phase_np.squeeze(0)
                    
                    est_chunk = processor.to_waveform(est_mag_np, mix_phase_np)
                    est_chunk = est_chunk[:chunk_end - chunk_start] # Trim padding
                
                # Overlap Add
                chunk_out_len = len(est_chunk)
                window = np.hanning(chunk_samples)[:chunk_out_len]
                
                est_full[chunk_start:chunk_start + chunk_out_len] += est_chunk * window
                weight_full[chunk_start:chunk_start + chunk_out_len] += window
                
                chunk_idx += 1
                pos += stride_samples
            
            print(" Done!")
            # Normalize
            est_full = np.divide(est_full, weight_full, where=weight_full > 0, out=est_full.copy())
            
            # Get spec for viz
            with torch.no_grad():
                est_spec_tuple = processor.to_spectrogram(est_full)
                est_mag_full = est_spec_tuple[0]
                while est_mag_full.dim() > 2: est_mag_full = est_mag_full.squeeze(0)
                if est_mag_full.shape[0] < est_mag_full.shape[1]: est_mag_full = est_mag_full.t()
                est_mag_np = est_mag_full.cpu().numpy()

            return est_full, est_mag_np

        # --- Run Inference ---
        print("\n" + "="*70)
        print("🔍 EVALUATING LSTM MODEL (Stage 2)")
        print("="*70)
        est_lstm, est_mag_lstm = predict_stem(model_lstm_s2, processor_lstm_s2, mix_segment)
        
        print("\n" + "="*70)
        print("🔍 EVALUATING U-NET MODEL (Stage 2)")
        print("="*70)
        est_unet, est_mag_unet = predict_stem(model_unet_s2, processor_unet_s2, mix_segment)
        
        # --- Visualization ---
        print("\n" + "="*70)
        print("SPECTROGRAMS COMPARISON (STAGE 2)")
        print("="*70)
        
        fig, axes = plt.subplots(3, 2, figsize=(16, 12))
        
        def display_spec(ax, data, title):
            im = ax.imshow(data, aspect='auto', origin='lower', cmap='viridis')
            ax.set_title(title, fontsize=12, fontweight='bold')
            ax.set_ylabel('Freq')
            ax.set_xlabel('Time')
            return im

        # Get Mix Mag
        mix_mag_tuple = processor_lstm_s2.to_spectrogram(mix_segment)
        mix_mag_viz = mix_mag_tuple[0].numpy()
        while mix_mag_viz.ndim > 2: mix_mag_viz = mix_mag_viz.squeeze(0)
        if mix_mag_viz.shape[0] < mix_mag_viz.shape[1]: mix_mag_viz = mix_mag_viz.T

        # Get Target Mag
        target_mag_tuple = processor_lstm_s2.to_spectrogram(target_segment)
        target_mag_viz = target_mag_tuple[0].numpy()
        while target_mag_viz.ndim > 2: target_mag_viz = target_mag_viz.squeeze(0)
        if target_mag_viz.shape[0] < target_mag_viz.shape[1]: target_mag_viz = target_mag_viz.T

        # Plot
        display_spec(axes[0, 0], mix_mag_viz, "Full Mixture (Input)")
        axes[0, 1].axis('off')
        axes[0, 1].text(0.5, 0.5, f"Stage 2: Full Mix -> Vocals\nSong: {selected_track.name}", 
                       ha='center', fontsize=12, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

        display_spec(axes[1, 0], target_mag_viz, "Target (Vocals)")
        display_spec(axes[1, 1], est_mag_lstm, "LSTM Prediction (Vocals)")
        display_spec(axes[2, 0], est_mag_unet, "U-Net Prediction (Vocals)")
        axes[2, 1].axis('off') 

        plt.tight_layout()
        plt.show()

        # --- Audio ---
        print("\n" + "="*70)
        print("AUDIO PLAYBACK (STAGE 2)")
        print("="*70)
        
        print("\n🔊 Full Mixture (Input):")
        display(Audio(mix_segment, rate=SR))
        
        print("\n🔊 Target (Vocals):")
        display(Audio(target_segment, rate=SR))
        
        print("\n🔊 LSTM Prediction:")
        display(Audio(est_lstm, rate=SR))
        
        print("\n🔊 U-Net Prediction:")
        display(Audio(est_unet, rate=SR))

    except Exception as e:
        print(f"❌ Error during inference: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⚠️  MUSDB18_PATH not set")

## Compare Training Results (Stage 2)

Side-by-side comparison of U-Net vs LSTM training curves for Stage 2.

In [ ]:
# ============================================================================
# PLOT TRAINING COMPARISON (STAGE 2)
# ============================================================================

if not STAGE2_ENABLED:
    print("⏭️  Stage 2 training disabled (STAGE2_ENABLED = False)")
else:
    if not hist_lstm_s2:
        ckpt_lstm_s2 = CHECKPOINT_DIR / f"model_a_lstm_stage2_{CHUNK_DURATION:.0f}s.pth"
        hist_lstm_s2 = utils.load_training_history_from_checkpoint(ckpt_lstm_s2)

    if not hist_unet_s2:
        ckpt_unet_s2 = CHECKPOINT_DIR / f"model_a_unet_stage2_{CHUNK_DURATION:.0f}s.pth"
        hist_unet_s2 = utils.load_training_history_from_checkpoint(ckpt_unet_s2)

    utils.plot_model_comparison(hist_lstm_s2, hist_unet_s2,
                                title="Model A Comparison: LSTM vs U-Net - Stage 2")

## Stage 2 Evaluation (Spectrograms + Audio)

Compare LSTM vs U-Net separation quality on Stage 2 test samples.

In [ ]:
# ============================================================================
# EVALUATE SPECTRO + AUDIO - STAGE 2 (4 -> 1)
# ============================================================================
# Goal: Test if the model can extract Vocals from the FULL mixture (Drums+Bass+Other+Vocals)

print(f"\n{'='*70}")
print('STAGE 2 EVALUATION: 4 → 1 (Full Mixture -> Vocals)')
print(f"{'='*70}\n")

# --- 1. Settings ---
EVAL_DURATION = 120.0  # <--- LIMIT: 2 Minutes (0:00 - 2:00)
START_OFFSET = 0.0

# --- 2. Load Model (Stage 2 Weights) ---
# We must manually initialize the "Light" architecture (32 filters, 5 layers)
model_eval = ma.TimeFrequencyDomainUNet(
    in_channels=1,
    out_channels=1,
    base_filters=32,   # Match training config
    num_layers=5,      # Match training config
    batchnorm=True,
    dropout=0.1
).to(device)

if ckpt_unet_s2.exists():
    checkpoint = torch.load(ckpt_unet_s2, map_location=device)
    model_eval.load_state_dict(checkpoint['model_state_dict'])
    print(f"✅ Loaded Stage 2 weights: {ckpt_unet_s2.name}")
else:
    print(f"❌ Stage 2 checkpoint not found! ({ckpt_unet_s2.name})")

model_eval.eval()

# --- 3. Run Inference on a Test File ---
# We use the 'test' split from the dataset
test_files = list((DATA_DIR / 'test').glob('*.wav')) # Adjust extension if needed (e.g., .stem.mp4 or directory structure)
if not test_files:
    # Fallback if specific test folder structure differs, try to pick from main data
    test_files = list(DATA_DIR.glob('*'))

if len(test_files) > 0:
    import random
    # Pick a random file
    test_file = test_files[0] # Fixed index for reproducibility, or use random.choice(test_files)
    print(f"📂 Processing: {test_file.name}")

    # A. Load Audio (Limit to 120s)
    # Note: We need the Full Mixture (Input) and the Vocals (Target)
    # Assuming standard MUSDB or similar structure where we can synthesize the mix
    # For this snippet, we'll rely on the utils to load/mix, but pass the duration.
    
    # We use the processor to load just the segment we want
    # (Assuming utils.load_test_sample handles the mixing of stems for Stage 2)
    # Stage 2 Input = Sum of All Stems (Mixture)
    # Stage 2 Target = Vocals
    
    # Custom Load Logic to ensure duration clamp:
    mix, target, sr = utils.load_test_sample_stage2(
        test_file, 
        duration=EVAL_DURATION, 
        offset=START_OFFSET,
        device=device
    )

    # B. Forward Pass
    with torch.no_grad():
        # Input shape: [1, 1, Freq, Time]
        mix_spec = processor_unet.wav_to_spectrogram(mix)
        
        # Predict
        est_spec = model_eval(mix_spec)
        
        # Invert to Audio
        est_audio = processor_unet.spectrogram_to_wav(est_spec)

    # --- 4. Visualize (No Error Map) ---
    import matplotlib.pyplot as plt
    import librosa.display

    # Convert to numpy for plotting (take first batch item, first channel)
    S_mix = mix_spec[0, 0].cpu().numpy()
    S_target = processor_unet.wav_to_spectrogram(target)[0, 0].cpu().numpy()
    S_est = est_spec[0, 0].cpu().numpy()

    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(18, 5)) # <--- ONLY 3 PLOTS
    
    # 1. Input (Mixture)
    ax = axes[0]
    img = librosa.display.specshow(S_mix, x_axis='time', y_axis='log', ax=ax, sr=sr)
    ax.set_title(f'Input: Full Mixture (0-{EVAL_DURATION}s)')
    fig.colorbar(img, ax=ax, format="%+2.0f dB")

    # 2. Target (Vocals)
    ax = axes[1]
    img = librosa.display.specshow(S_target, x_axis='time', y_axis='log', ax=ax, sr=sr)
    ax.set_title(f'Target: Vocals (Ground Truth)')
    fig.colorbar(img, ax=ax, format="%+2.0f dB")

    # 3. Estimate (Separated)
    ax = axes[2]
    img = librosa.display.specshow(S_est, x_axis='time', y_axis='log', ax=ax, sr=sr)
    ax.set_title(f'Prediction: Separated Vocals')
    fig.colorbar(img, ax=ax, format="%+2.0f dB")

    plt.tight_layout()
    plt.show()

    # --- 5. Listen ---
    print("🎧 Input Mixture (Full Band):")
    utils.play_audio(mix, sr)
    
    print("🎧 Target Vocals:")
    utils.play_audio(target, sr)
    
    print("🎧 Separated Vocals (Model Output):")
    utils.play_audio(est_audio, sr)

else:
    print("⚠️ No test files found in data directory.")

## Quantitative Evaluation

Compute BSS metrics (SDR/SIR/SAR) on test set using museval library.

In [ ]:
# ============================================================================
# QUANTITATIVE EVALUATION - BSS METRICS
# ============================================================================

NUM_TEST_SAMPLES = 10
STAGE_FOR_EVAL = "stage2" if STAGE2_ENABLED else "stage1"

# Evaluate both models using utility function
metrics = utils.evaluate_separation_quality(
    model_lstm=model_lstm,
    model_unet=model_unet,
    processor_lstm=processor_lstm,
    processor_unet=processor_unet,
    test_data_dir=DATA_DIR,
    stage=STAGE_FOR_EVAL,
    num_samples=NUM_TEST_SAMPLES,
    sr=22050,
    device=device
)

## Optional: Custom Song Inference (Upload)

Upload a song (or place it in the folder below) and run inference with both models. This is the final step.

In [ ]:
# ============================================================================
# CUSTOM SONG INFERENCE (UPLOAD OR LOCAL FOLDER)
# ============================================================================

UPLOAD_DIR = DATA_DIR / "user_uploads"
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

# Colab upload
if IN_COLAB:
    try:
        from google.colab import files
        uploaded = files.upload()
        for name, data in uploaded.items():
            out_path = UPLOAD_DIR / name
            with open(out_path, "wb") as f:
                f.write(data)
            print(f"✅ Saved: {out_path}")
    except Exception as e:
        print(f"⚠️  Colab upload failed: {e}")
        print(f"Place a file manually in: {UPLOAD_DIR}")
else:
    print(f"📁 Place your audio file in: {UPLOAD_DIR}")

# Pick the most recent audio file
exts = ["*.wav", "*.mp3", "*.flac", "*.ogg", "*.m4a"]
audio_files = []
for ext in exts:
    audio_files += list(UPLOAD_DIR.glob(ext))

audio_files = sorted(audio_files, key=lambda p: p.stat().st_mtime, reverse=True)

if not audio_files:
    print("⚠️  No audio files found in upload folder.")
else:
    audio_path = audio_files[0]
    print(f"🎵 Using file: {audio_path.name}")

    utils.compare_models_on_audio_file(
        file_path=audio_path,
        model_lstm=model_lstm,
        model_unet=model_unet,
        processor_lstm=processor_lstm,
        processor_unet=processor_unet,
        device=device,
        sr=22050,
        duration=15  # seconds (set None for full length)
    )